In [5]:
import numpy
import matplotlib
import nltk

nltk.download('brown') # corpus
nltk.download('universal_tagset') # simplified tag set

# loading the corpus
tagged_sentences = nltk.corpus.brown.tagged_sents(tagset='universal')

print(f"Total sentences: {len(tagged_sentences)}")
print(f"Example sentence: {tagged_sentences[0]}")

[nltk_data] Downloading package brown to
[nltk_data]     /Users/josephinebhadran/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /Users/josephinebhadran/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Total sentences: 57340
Example sentence: [('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [7]:
import random

random.seed(1234)
shuffled = list(tagged_sentences)
random.shuffle(shuffled)

# train test split
split = int(0.8 * len(shuffled))
train_sents = shuffled[:split]
test_sents = shuffled[split:]


print(f"training: {len(train_sents)} sentences")
print(f"testing: {len(test_sents)} sentences")

training: 45872 sentences
testing: 11468 sentences


In [13]:
# training
from collections import defaultdict


def train_hmm(sentences):
    transition_counts = defaultdict(lambda: defaultdict(int)) # dict for how many times tag B followed tag A
    emission_counts = defaultdict(lambda: defaultdict(int)) # dict for how many times this word appeared with this tag
    tag_counts = defaultdict(int) # total times this tag appeared


    for sentence in sentences:
        prev_tag = "<START>"

        for word, tag in sentence:
            word = word.lower()

            tag_counts[tag] += 1
            emission_counts[tag][word] += 1
            transition_counts[prev_tag][tag] += 1

            prev_tag = tag

        transition_counts[prev_tag]["<END>"] += 1

    
    # sanity check
    print(f"Tags found: {list(tag_counts.keys())}")
    print(f"Example emissions for NOUN: {list(emission_counts['NOUN'].items())[:5]}")

    return transition_counts, emission_counts, tag_counts


def compute_probabilities(transition_counts, emission_counts, tag_counts):
    transition_probs = defaultdict(lambda: defaultdict(float))
    emission_probs = defaultdict(lambda: defaultdict(float))

    # transition probabilities
    for prev_tag in transition_counts:
        total = sum(transition_counts[prev_tag].values())
        for tag in transition_counts[prev_tag]:
            transition_probs[prev_tag][tag] = transition_counts[prev_tag][tag] / total

    # emission probabilities
    for tag in emission_counts:
        total = tag_counts[tag]
        for word in emission_counts[tag]:
            emission_probs[tag][word] = emission_counts[tag][word] / total

    return transition_probs, emission_probs


In [14]:
transition_counts, emission_counts, tag_counts = train_hmm(train_sents)
transition_probs, emission_probs = compute_probabilities(transition_counts, emission_counts, tag_counts)

# sanity check
print(f"Transition probs from START: {dict(transition_probs['<START>'])}")
print(f"P(runs | VERB): {emission_probs['VERB']['runs']}")


Tags found: ['VERB', 'NOUN', 'ADP', 'DET', 'ADV', 'ADJ', '.', 'PRON', 'CONJ', 'PRT', 'NUM', 'X']
Example emissions for NOUN: [('nothing', 327), ('country', 254), ("tourists'", 1), ('comfort', 33), ('shoulders', 43)]
Transition probs from START: {'VERB': 0.04534356470177886, 'PRON': 0.15857167771189395, '.': 0.08931374258807115, 'NOUN': 0.14167683990233693, 'ADP': 0.12360481339379142, 'DET': 0.21267875828392047, 'PRT': 0.036645448203697244, 'NUM': 0.016502441576560865, 'CONJ': 0.048831531217300314, 'ADJ': 0.034247471224276245, 'ADV': 0.09210411580048831, 'X': 0.00047959539588419953}
P(runs | VERB): 6.84439273125492e-05
